In [1]:
# !pip install hurst
# !pip install arch 
# !pip install lightgbm 
# !pip install scikit-learn 
# !pip install pmdarima
# !pip install shap
# !pip install numpy pandas
# Fed API: 1f3befec93b6c68383d27f3b58cca038
# !pip install fredapi
# 	Final_Historical_Score
# CNY	0.74. 1
# INR	0.69. 2
# AUD	0.57
# CAD	0.54. 10
# CHF	0.52
# USD	0.48. 6
# JPY	0.46. 4
# GBP	0.42. 5
# EUR	0.38. 9. Ger is 3. Ital is 8. Fra is 9

In [2]:
# Import the libraries
import pyautogui
import time
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import os
import itertools

from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)  # or None for no limit

In [5]:
#To run the modules created
import sys
sys.path.append("..")                 # if needed to find project_pkg
import project_pkg.dld as dld  # import module object
import project_pkg.dld_h4 as dh4 # import module object
from importlib import reload
reload(dld)
reload(dh4)

# call function/class inside the module
# Define the asset lists according to your exact naming conventions
PRIMARY_ASSETS = ["USDJPY", "EURUSD", "US30.a", "XAUUSD.a", "NAS100.a", "USDINR"]
SECONDARY_ASSETS = ["USTN10YR-F.a", "SpotBrent", "VIX.a", "USDX.a"]
ALL_ASSETS = PRIMARY_ASSETS + SECONDARY_ASSETS
filtered_df = pd.DataFrame({'Currency': ['AUDUSD', *ALL_ASSETS]})

# dld.dld(ALL_ASSETS)
# dh4.dld(PRIMARY_ASSETS)

import project_pkg.utils as utils

In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

# 1. Import the unified high-level cockpit and support constants
from project_pkg.model_new import (
    evaluate_symbol_cockpit,
    PRIMARY_ASSETS
)

# Silence diagnostic warnings
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
warnings.filterwarnings("ignore", message=".*No supported index is available.*")

# --- STEP 1: PRE-LOAD ALL ASSET FILES INTO A DICTIONARY FOR CROSS-ASSET CONTEXT ---
print("Pre-loading cross-asset daily context network...")
dfs_dict = {}
for asset in ALL_ASSETS:
    file_path = fr"C:\Users\nitis\Documents\Forex\Data\New data\{asset}.csv"
    try:
        df_temp = pd.read_csv(file_path, sep='\t', engine='python')
        df_temp["<DATE>"] = df_temp["<DATE>"].astype(str)
        dfs_dict[asset] = df_temp
    except Exception as e:
        print(f" -> Warning: Could not pre-load {asset} context. Error: {e}")

print("Cross-asset network mapping completed.\n")

# --- STEP 2: PIPELINE PROCESS LOOP FOR PRIMARY ASSETS ONLY ---
for currency in filtered_df['Currency']:
    if pd.isna(currency):
        continue
    
    # CRITICAL FILTER: Skip evaluation and prediction of secondary assets
    if currency not in PRIMARY_ASSETS:
        print(f"Skipping prediction pipeline for Secondary Asset context holder: {currency}")
        print("-" * 115)
        continue
        
    file_path = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}.csv"
    file_path_h4 = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}_h4.csv"
    
    try:
        df_mod = pd.read_csv(file_path, sep='\t', engine='python')
        df_mod_h4 = pd.read_csv(file_path_h4, sep='\t', engine='python')
        df_mod["<DATE>"] = df_mod["<DATE>"].astype(str)
        df_mod_h4["<DATE>"] = df_mod_h4["<DATE>"].astype(str)
    except Exception as e:
        print(f"Skipping {currency}: Failed to load file data. Error: {e}")
        continue

    # Guard clause for dataset window boundaries
    if len(df_mod) < 800:
        print(f"Skipping {currency}: Historical dataset size ({len(df_mod)} rows) insufficient.")
        continue

    # --- RUN SIMPLIFIED EXECUTIVE COCKPIT ---
    # This runs the backtests, profiles the volumes, checks alignment, 
    # runs stress tests, and prints exactly your 4-line executive summary.
    evaluate_symbol_cockpit(
        df_daily=df_mod,
        df_h4=df_mod_h4,
        train_window=700,
        target_symbol=currency,
        dfs_dict=dfs_dict,
        show_deep_dive=False # Flip to True if you need to troubleshoot a specific symbol's weights
    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pre-loading cross-asset daily context network...
Cross-asset network mapping completed.


 EXECUTIVE COCKPIT SUMMARY: AUDUSD
0. CONCLUSION        : STABLE MIXED REGIME: Market indicators are running inside historical baseline trends. Standard allocation thresholds apply.
1. ENGINE ALIGNMENT  : CONCENTRATED (Same Direction Risk | E1 Coef: +0.90, E2 Coef: +1.95)
2. SCENARIO REGIMES  : Currently mapping as [Breakout->Compressed Range, Chop->Compressed Range]
3. VOLUME TRACKING   : NORMAL (Tracking at 1.1x normal volume - Baseline Grid)
4. STRESS PERF RUN   : Steady tracking across simulated tail-risks: Liquidity Shock (Meta-MAE: 0.0045), Bear Grind (Meta-MAE: 0.0148), Bull Bubble (Meta-MAE: 1.9720)

 EXECUTIVE COCKPIT SUMMARY: USDJPY
0. CONCLUSION        : STABLE MIXED REGIME: Market indicators are running inside historical baseline trends. Standard allocation thresholds apply.
1. ENGINE ALIGNMENT  : C

# ==========================================================================================
#                         UNDERSTANDING THE PORTFOLIO COCKPIT METRICS
# ==========================================================================================

### 0. EXECUTIVE CONCLUSION STATE MATRIX (One-Line Summary Options)
This metric automatically evaluates the interaction of **Volume Tracking**, **Core Alignment**, and **Scenario Regimes** to provide a high-level tactical vector:

* 🔴 **WARNING: High-Risk Volatile Breakout** * *Trigger:* Volume is **HIGH** + Regimes map to **High-Vol Expansion** or **High-Vol Grind**.
    * *Tactical Focus:* Protect risk boundaries, expect wider swings; momentum models dominant.
* 🟢 **BULLISH EXPANSION: Pure Momentum Play**
    * *Trigger:* Regimes map to **High-Vol Expansion** or **Mature Trend** (with normal volume).
    * *Tactical Focus:* Strong structural trend detected. Highly conducive to Trend-Following vectors.
* 🔵 **STABLE MEAN-REVERSION: Range/Fade Strategy**
    * *Trigger:* Engine Alignment is **OPTIMAL (Hedged)** + Regimes map to **High-Vol Grind** or **Compressed Range**.
    * *Tactical Focus:* Range constraints bound. Engines are hedging each other efficiently. Favor Grid/Fading styles.
* 🟡 **COMPRESSION HOLD: Wait-and-See**
    * *Trigger:* Volume is **LOW** + Regimes map to **Compressed Range**.
    * *Tactical Focus:* Extreme liquidity squeeze. Volatility is coiled tight; standby for an explosive breakout.
* ⚪ **STABLE MIXED REGIME: Baseline Alignment**
    * *Trigger:* Baseline indicators (Normal volume, standard regimes).
    * *Tactical Focus:* Market indicators are running inside historical norms. Standard allocation thresholds apply.

---

### Core Metric Glossary

| Metric Name | Status/Outputs | Practical Meaning for Portfolio Manager |
| :--- | :--- | :--- |
| **1. ENGINE ALIGNMENT** | `OPTIMAL (Hedged)` <br> `CONCENTRATED` | **Are our core engines hedging each other?** <br> • **Hedged**: Mean-Reversion and Momentum models are pulling in opposite directions (safer, self-balancing). <br> • **Concentrated**: Both models agree on the trend (higher directional exposure, higher capital risk). |
| **2. SCENARIO REGIMES** | `High-Vol Expansion` <br> `High-Vol Grind` <br> `Mature Trend` <br> `Compressed Range` | **What is the current "mood" of the asset's structure?** <br> Classified by an unsupervised K-Means clustering algorithm looking at short-term standard deviation, distance-to-MA, True Range, and volume ratio. |
| **3. VOLUME TRACKING** | `HIGH` (>1.2x) <br> `LOW` (<0.8x) <br> `NORMAL` | **Are institutions actively participating right now?** <br> Measures current daily session volume against its 20-day trailing moving average. High volume validates trends; low volume warns of a coil/squeeze. |
| **4. STRESS PERF RUN** | `Mean Absolute Error (MAE)` | **How did the stacked meta-model survive simulated tail-risk history?** <br> Shows OOS (Out-of-Sample) errors under Liquidity Black Holes, Structural Bear Markets, and Speculative Bubbles. Lower MAE = higher model resilience. |